<!-- nav -->
[← 7 · Bindings](07_Bindings.ipynb) · [Home](../../README.md) · [10 · Answers you can trust →](10_Answers_You_Can_Trust.ipynb)

# 9 · Constraints as data

**The problem.** A shipment of vaccine must stay between 2 and 8 °C. That constraint lives in the QA team's spreadsheet,
in the Python pipeline that processes the data loggers' uploads, in the TypeScript dashboard that flags shipments, and
in the logger's firmware, which sounds an alarm. Four copies, in four languages. When the range changes for a new
product, someone updates three of them.

A constraint in code can't be stored, sent or read by another language: a TypeScript function means nothing to the
Python pipeline. mbse-expressions writes the constraint once, as an **expression**: data with a schema, like any
mbse-schemas object, that every program loads and evaluates the same way, and that translates into each of those
languages (case study 14).

This case study writes a constraint, evaluates it, checks it, and saves it. The cold chain's schemas are in
`toolkit.ts`, beside this notebook: a `Shipment` has the latest temperature its logger reported (`celsius`), every
reading (`readings`), the hours it has been in transit (`hours`), and the loggers it carries. `cold_chain()` gives an
mbse-schemas store of them, which holds expressions too. Objects are written in a store within a transaction.

In [1]:
import { Evaluators, Expressions as E } from "@mbse/expressions/Patterns/Expressions";
import { Terms } from "@mbse/expressions/Framework";
import { JSON as SchemaJSON } from "@mbse/schemas/Framework";
import { cold_chain, show } from "./toolkit.js";

const store = cold_chain();
const Expression = store.Expressions.Framework.Expression;
const [s1, s2] = store.transaction(() => [
  store.Shipment((s: any) => s.id("S1").celsius(5.5).hours(30n)),
  store.Shipment((s: any) => s.id("S2").celsius(9.1).hours(80n))]);

<a id="T09.1"></a>
## 09.1 · Writing a constraint

Constraints are built with **writers**, the shorthand for the builders' calls that [tutorial 3](03_The_Smallest_Expression_A_Literal.ipynb#T03.1)
introduces, with what an expression, its `root` and its terms are. `E.variable("shipment")` is the variable the
constraint is about, the shipment; reading a property of its writer reads that property, and methods build calls. The
constraint is an `Expression`, written within a transaction, as the shipments were:

In [2]:
const shipment = E.variable("shipment");
const celsius = shipment.celsius;
const in_range = store.transaction(() => Expression((e: any) => e.root(E.literal(2.0).le(celsius).and_(celsius.le(8.0)))));
console.log(show(in_range));

and(le(2.0, get(shipment, 'celsius')), le(get(shipment, 'celsius'), 8.0))


`show` (from the toolkit) writes the expression as function calls. `shipment.celsius` reads a property: `get`. `le` is
"less than or equal", and the two comparisons are joined by `and`. These names are the **core vocabulary** that every
program evaluating expressions implements, in TypeScript and in Python alike.

<a id="T09.2"></a>
## 09.2 · Evaluating it

`Evaluators.OfAny(constraint, scope)` computes a constraint's value, with the variables in the scope bound to values:

In [3]:
console.log(Evaluators.OfAny(in_range, { shipment: s1 }), Evaluators.OfAny(in_range, { shipment: s2 }));

true false


Checking a constraint against each shipment is evaluating it with `shipment` bound to each in turn:

In [4]:
console.log([s1, s2].map((s: any) => Evaluators.OfAny(in_range, { shipment: s })));

[ true, false ]


<a id="T09.3"></a>
## 09.3 · What a constraint can say

A constraint can have several variables, and use values from your program. A value like `MAX_HOURS` becomes a
**literal** when the constraint is written: the expression doesn't refer back to your program, so it means the same
thing wherever it's loaded. Integers are `bigint`s, as they are in mbse-schemas: `72n`, not `72`, which is a float.

In [5]:
const MAX_HOURS = 72n;
const [on_time, late_by] = store.transaction(() => [
  Expression((e: any) => e.root(shipment.hours.le(MAX_HOURS))),
  Expression((e: any) => e.root(shipment.hours.add(E.variable("now")).gt(MAX_HOURS)))]);
console.log(show(on_time));
console.log(show(late_by));
console.log(Evaluators.OfAny(late_by, { shipment: s1, now: 50n }));

le(get(shipment, 'hours'), 72)


gt(add(get(shipment, 'hours'), now), 72)


true


To name an intermediate value, use a **let**: the name is bound to the value inside the body, and nowhere else.

In [6]:
const temperature = E.variable("c");
const named = store.transaction(() =>
  Expression((e: any) => e.root(E.let_("c", shipment.celsius, temperature.ge(2.0).and_(temperature.le(8.0))))));
console.log(show(named));

let c = get(shipment, 'celsius') in and(ge(c, 2.0), le(c, 8.0))


Writers build only what the core vocabulary has. Patterns' operations are data, which its dialect defines: each one an
expression schema whose variables are its arguments, in order, and whose result is its return. A call outside them can
still be written, with `E.operation(name, ...arguments)`, but validation refuses it. The core has no division, because
division by zero would make a constraint partial
([CAP-095: Basic terms as data](https://github.com/pitaman71/mbse-basis/blob/main/docs/capabilities/CAP-095.md#terms-and-the-core-vocabulary)).

<a id="T09.4"></a>
## 09.4 · Validating a constraint before it is evaluated

A constraint that arrives as data, from a file or another program, may be broken. `E.DIALECT.validate` reports what
evaluating it would raise: variables nothing binds, wrong numbers of arguments, and calls to an operation Patterns doesn't
define.

In [7]:
const halved = store.transaction(() => Expression((e: any) => e.root(E.operation("div", shipment.celsius, 2.0))));
console.log(E.DIALECT.validate(late_by, ["shipment", "now"]));
console.log(E.DIALECT.validate(late_by, ["shipment"]));
console.log(E.DIALECT.validate(halved, ["shipment"]));

[]


[ "argument 0: argument 1: variable 'now' is not bound" ]


[ "'div' is not an operation Patterns defines" ]


<a id="T09.5"></a>
## 09.5 · Saving and loading

An expression's term is a value of the schema `Expressions.Framework.Term`, which the store holds, so it saves like any
mbse-schemas value. `Terms.term(constraint)` reads an expression's root term:

In [8]:
const Term = store.registered("Expressions.Framework.Term");
const text = SchemaJSON.ToJSON(store)(Term, Terms.term(in_range));
console.log(SchemaJSON.ToJSON(store)(Term, Terms.term(in_range), { indent: 1 })); // indented, to read

{
 "patterns": {
  "call": {
   "callable": "and",
   "arguments": [
    {
     "patterns": {
      "call": {
       "callable": "le",
       "arguments": [
        {
         "patterns": {
          "literal": {
           "value": {
            "float": 2.0
           }
          }
         }
        },
        {
         "patterns": {
          "call": {
           "callable": "get",
           "arguments": [
            {
             "patterns": {
              "variable": {
               "name": "shipment"
              }
             }
            },
            {
             "patterns": {
              "literal": {
               "value": {
                "str": "celsius"
               }
              }
             }
            }
           ]
          }
         }
        }
       ]
      }
     }
    },
    {
     "patterns": {
      "call": {
       "callable": "le",
       "arguments": [
        {
         "patterns": {
          "call": {
           "callable": "get"

Each term is written as its dialect, `patterns`, then its kind, `call`, `literal` or `variable`, holding its
properties: a call's `callable` and its `arguments`, in order. A term is a value, not an object of its own, so `get(shipment, 'celsius')`, which the constraint uses
twice, is written at each use. A literal's value can be any value, so it says which schema it holds (`"float"`,
`"str"`). Nothing in the text is TypeScript: it's byte for byte the text the Python tutorial writes,
and the Python pipeline reads it and evaluates the same constraint. Here it's written back as the root of a new
expression:

In [9]:
const loaded = store.transaction(() => Expression((e: any) => e.root(SchemaJSON.FromJSON(store)(Term, text))));
console.log(show(loaded), Terms.same(loaded, in_range));
console.log([s1, s2].map((s: any) => Evaluators.OfAny(loaded, { shipment: s })));

and(le(2.0, get(shipment, 'celsius')), le(get(shipment, 'celsius'), 8.0)) true


[ true, false ]


## Why it works this way

- **A constraint is data, not code.** Data can be stored beside what it's about, versioned, diffed, sent, and loaded by
  a program in another language. Code can only be run, and only by its own language
  ([FEA-024: expressions as data](https://github.com/pitaman71/mbse-basis/blob/main/docs/features/FEA-024.md)).
- **A small core vocabulary.** Its operations are pure, total and bounded, so every implementation evaluates them the
  same way, and validation refuses a call to any other
  ([CAP-095: Basic terms as data](https://github.com/pitaman71/mbse-basis/blob/main/docs/capabilities/CAP-095.md#terms-and-the-core-vocabulary)).
- **Values become literals.** A constraint that referred back into the program that wrote it couldn't be loaded anywhere
  else.

The next case study looks at what the constraints answer when the data is incomplete, which is most of the time.

<!-- nav -->
[← 7 · Bindings](07_Bindings.ipynb) · [Home](../../README.md) · [10 · Answers you can trust →](10_Answers_You_Can_Trust.ipynb)